# Fundamento matemático: velocidad cartesiana máxima sostenible vía Jacobiano

**Contexto**: robot ABB IRB 140 reducido a los primeros 3 ejes. Se busca la trayectoria recta de longitud fija $L$ que puede recorrerse a la mayor **velocidad cartesiana constante** sin violar los límites de velocidad articular.

Este documento formaliza el funcional de tiempo mínimo y lo conecta con la teoría del Jacobiano posicional.

---
## §1 — Planteo del problema

Sea $\mathbf{p}_0, \mathbf{p}_1 \in \mathbb{R}^3$ el punto inicial y final de una trayectoria recta. Se define:

$$L = \|\mathbf{p}_1 - \mathbf{p}_0\|_2 \qquad \hat{\mathbf{d}} = \frac{\mathbf{p}_1 - \mathbf{p}_0}{L}$$

La parametrización temporal con velocidad cartesiana **constante** $v > 0$ es:

$$\mathbf{p}(t) = \mathbf{p}_0 + v\,t\,\hat{\mathbf{d}}, \qquad t \in \left[0,\, T\right], \quad T = \frac{L}{v}$$

**Pregunta**: ¿cuál es el máximo $v$ admisible, y en qué posición $(y_0, z_0)$ del plano frontal se alcanza?

---
## §2 — Cinemática diferencial para los primeros 3 ejes

Sea $\mathbf{q} = (q_1, q_2, q_3)^\top \in \mathbb{R}^3$ el vector articular. La cinemática diferencial es:

$$\dot{\mathbf{p}} = J_{pos}(\mathbf{q})\,\dot{\mathbf{q}}$$

donde $J_{pos}(\mathbf{q}) \in \mathbb{R}^{3\times 3}$ es el **Jacobiano posicional** — las primeras 3 filas y 3 columnas del Jacobiano geométrico completo $J_0 \in \mathbb{R}^{6\times 3}$:

$$J_{pos} = J_0[\{0,1,2\},\, \{0,1,2\}]$$

Para velocidad cartesiana constante $\dot{\mathbf{p}} = v\,\hat{\mathbf{d}}$, las velocidades articulares requeridas son:

$$\dot{\mathbf{q}}(t) = J_{pos}^{-1}(\mathbf{q}(t))\; v\,\hat{\mathbf{d}}$$

> **Nota**: $J_{pos}$ es cuadrada ($3\times3$) porque hay exactamente 3 ejes y 3 grados de libertad de posición. Esto garantiza (fuera de singularidades) que la inversa existe.

---
## §3 — Velocidad máxima factible en una configuración

Los límites articulares imponen:

$$|\dot{q}_i(t)| \leq \dot{q}_{i,\max}, \qquad i = 1, 2, 3$$

Sustituyendo la expresión de $\dot{\mathbf{q}}$:

$$v\, \left|[J_{pos}^{-1}(\mathbf{q})\,\hat{\mathbf{d}}]_i\right| \leq \dot{q}_{i,\max}$$

La velocidad máxima admisible **en la configuración $\mathbf{q}$** y **dirección $\hat{\mathbf{d}}$** es:

$$\boxed{v_{\max}(\mathbf{q}) = \frac{1}{\displaystyle\max_{i=1,2,3}\; \frac{\left|[J_{pos}^{-1}(\mathbf{q})\,\hat{\mathbf{d}}]_i\right|}{\dot{q}_{i,\max}}}}$$

Usando la notación de norma $\ell^\infty$ ponderada con la matriz de escala $D_{\max} = \mathrm{diag}(\dot{q}_{1,\max}, \dot{q}_{2,\max}, \dot{q}_{3,\max})$:

$$v_{\max}(\mathbf{q}) = \left\|D_{\max}^{-1}\,J_{pos}^{-1}(\mathbf{q})\,\hat{\mathbf{d}}\right\|_\infty^{-1}$$

---
## §4 — Funcional de tiempo mínimo

Para mantener **velocidad constante**, el robot debe cumplir los límites en **toda** la trayectoria simultáneamente. La velocidad máxima sostenible es el mínimo de $v_{\max}$ sobre el recorrido:

$$v^* = \min_{s\in[0,1]} v_{\max}(\mathbf{q}(s))$$

donde $\mathbf{q}(s)$ es la solución de cinemática inversa en el punto $\mathbf{p}(s) = \mathbf{p}_0 + s(\mathbf{p}_1-\mathbf{p}_0)$.

El **tiempo mínimo** para recorrer el segmento a velocidad constante es:

$$\boxed{T^* = \frac{L}{v^*} = L\cdot\max_{s\in[0,1]}\left\|D_{\max}^{-1}\,J_{pos}^{-1}(\mathbf{q}(s))\,\hat{\mathbf{d}}\right\|_\infty}$$

Este es el **funcional a minimizar** sobre el conjunto de trayectorias rectas factibles.

### Relación con la discretización numérica

El algoritmo de `analizar_tiempos_trayectoria` en el notebook principal calcula exactamente el numerador de $T^*$:

```
dt_constante = max( |Δq| / qd_max )   # max sobre todos los segmentos y articulaciones
T* ≈ N · dt_constante
```

donde $\Delta q_{k,i} = q_{k+1,i} - q_{k,i}$ es la variación articular entre pasos consecutivos (aproxima $J_{pos}^{-1}\hat{d}\,\Delta s\,L$), y $N$ es el número de segmentos. La equivalencia es exacta en el límite $N \to \infty$.

---
## §5 — Conexión con los valores singulares: elipsoide de manipulabilidad

La descomposición en valores singulares del Jacobiano posicional es:

$$J_{pos} = U\Sigma V^\top, \qquad \Sigma = \mathrm{diag}(\sigma_1, \sigma_2, \sigma_3), \quad \sigma_1 \geq \sigma_2 \geq \sigma_3 \geq 0$$

Luego $J_{pos}^{-1} = V\Sigma^{-1}U^\top$ y $\|J_{pos}^{-1}\|_2 = \sigma_3^{-1}$.

### Cota inferior del tiempo

Para el caso de límites articulares isótropos ($\dot{q}_{i,\max} = \dot{q}_{\max}$ para todo $i$), se tiene:

$$\left\|D_{\max}^{-1}J_{pos}^{-1}\hat{\mathbf{d}}\right\|_\infty \leq \left\|J_{pos}^{-1}\hat{\mathbf{d}}\right\|_2 \cdot \frac{1}{\dot{q}_{\max}} \leq \frac{1}{\sigma_3(J_{pos})\,\dot{q}_{\max}}$$

Por lo tanto:

$$T^* \geq \frac{L}{\sigma_3(J_{pos,\min})\,\dot{q}_{\max}}$$

donde $\sigma_3(J_{pos,\min})$ es el mínimo valor singular mínimo a lo largo de la trayectoria.

### Elipsoide de manipulabilidad

El conjunto de velocidades cartesianas alcanzables con $\|\dot{\mathbf{q}}\|_2 \leq 1$ forma la elipsoide:

$$\mathcal{E}(\mathbf{q}) = \left\{\mathbf{v} \in \mathbb{R}^3 : \mathbf{v}^\top (J_{pos}J_{pos}^\top)^{-1}\mathbf{v} \leq 1\right\}$$

- Los **semiejes** tienen longitud $\sigma_1 \geq \sigma_2 \geq \sigma_3$.
- La **dirección de máxima velocidad** es $\mathbf{u}_1$ (vector singular izquierdo asociado a $\sigma_1$).
- La **dirección de mínima velocidad** es $\mathbf{u}_3$ (asociada a $\sigma_3$).
- En una **singularidad**: $\sigma_3 \to 0$ y la elipsoide degenera en un disco — hay al menos una dirección cartesiana que no puede seguirse.

Por eso el algoritmo usa $\sigma_{min}$ como métrica secundaria: ante igualdad de tiempo, prefiere la trayectoria alejada de singularidades.

---
## §6 — Problema de optimización sobre la grilla frontal

Se busca el par $(y_0, z_0)$ que resuelve:

$$\min_{(y_0, z_0) \in \mathcal{F}} T^*(y_0, z_0) = \min_{(y_0, z_0) \in \mathcal{F}} L\cdot\max_{s\in[0,1]} \left\|D_{\max}^{-1}\,J_{pos}^{-1}(\mathbf{q}(s;\,y_0,z_0))\,\hat{\mathbf{d}}\right\|_\infty$$

donde $\mathcal{F}$ es el conjunto de trayectorias factibles (dentro del workspace, sin singularidades, longitud $L = 20$ cm, dirección $\hat{\mathbf{d}} = \hat{x}$).

### ¿Por qué no se puede resolver analíticamente?

1. **$\mathbf{q}(s; y_0, z_0)$ no tiene forma cerrada**: la cinemática inversa del IRB 140 implica relaciones trigonométricas no triviales (aunque tiene solución analítica, el seguimiento de rama continua a lo largo de la trayectoria no).

2. **El funcional es un $\max$ sobre $s$**: no es diferenciable en general — el problema es de tipo minimax no suave.

3. **La función $(y_0, z_0) \mapsto T^*$ es no convexa**: tiene múltiples mínimos locales asociados a distintas ramas de la IK.

**Conclusión**: la búsqueda exhaustiva en grilla es la estrategia correcta para este tamaño de problema. Para grillas más densas o robots de más ejes se pueden usar métodos de optimización sin gradiente (Nelder-Mead, algoritmos evolutivos) o diferenciación automática a través de solvers IK.

---
## §7 — Equivalencia entre el funcional y el algoritmo numérico

El algoritmo de `barrer_plano_frontal` implementa la siguiente aproximación discreta:

| Objeto continuo | Aproximación discreta |
|---|---|
| $\mathbf{p}(s) = \mathbf{p}_0 + s\,L\hat{d}$ | `np.linspace(p_inicio, p_fin, N)` |
| $\mathbf{q}(s)$ vía IK | `resolver_ik_posicion` paso a paso (warm-start) |
| $\dot{\mathbf{q}} = J^{-1}\dot{\mathbf{p}}$ | $\Delta\mathbf{q}_k / \Delta t$ (diferencia finita) |
| $\max_s \|D^{-1}J^{-1}\hat{d}\|_\infty$ | `np.max(|Δq| / qd_max)` sobre todos los segmentos |
| $T^* = L \cdot (\cdots)$ | `N · dt_constante` |

La aproximación converge cuando $N \to \infty$. Con $N = 35$ pasos y $L = 20$ cm el paso cartesiano es $\approx 5{,}9$ mm, suficiente para capturar la variación del Jacobiano en el workspace del IRB 140.

### Criterio de selección de la mejor trayectoria

El algoritmo selecciona la trayectoria con:
$$\arg\min_{(y_0, z_0)} T^*(y_0, z_0)$$
y, en caso de empate numérico ($\Delta T < 10^{-9}$ s):
$$\arg\max_{(y_0, z_0)} \min_{s} \sigma_3(J_{pos}(\mathbf{q}(s; y_0, z_0)))$$

Este segundo criterio favorece trayectorias alejadas de singularidades, que son más robustas frente a errores de posicionamiento.

---
## §8 — Ejemplo numérico ilustrativo

Se construye el robot, se evalúa $v_{\max}(\mathbf{q})$ y $\sigma_{min}(J)$ a lo largo de una trayectoria de muestra, y se compara con la fórmula analítica.

In [ ]:
# Si se ejecuta en Colab, descomentar:
# %pip install roboticstoolbox-python
# %pip install numpy==1.26.4 --force-reinstall

import numpy as np
import matplotlib.pyplot as plt
import roboticstoolbox as rtb
import spatialmath as sm

QD_MAX = np.deg2rad([200.0, 200.0, 260.0])  # [rad/s]


def crear_robot():
    link1 = rtb.RevoluteDH(alpha=-np.pi / 2, a=0.07, d=0.352)
    link2 = rtb.RevoluteDH(a=0.36, offset=-np.pi / 2)
    link3 = rtb.RevoluteDH(alpha=np.pi / 2, offset=np.pi)
    link4 = rtb.RevoluteDH(d=0.38, alpha=-np.pi / 2)
    link5 = rtb.RevoluteDH(alpha=np.pi / 2)
    link6 = rtb.RevoluteDH(d=0.065)
    tool = link4.A(0.0) * link5.A(0.0) * link6.A(0.0)
    robot = rtb.DHRobot([link1, link2, link3], tool=tool, name="IRB140_3EJES")
    robot.qlim = np.deg2rad([[-180, -100, -220], [180, 100, 60]])
    return robot


robot = crear_robot()
print("Robot creado:", robot.name)

In [ ]:
def v_max_en_configuracion(robot, q, d_hat, qd_max):
    """Velocidad cartesiana máxima en dirección d_hat dado el límite articular."""
    Jpos = np.asarray(robot.jacob0(q)[:3, :3], dtype=float)
    try:
        Jinv = np.linalg.inv(Jpos)
    except np.linalg.LinAlgError:
        return 0.0
    needed = Jinv @ d_hat           # velocidad articular por unidad de velocidad cartesiana
    ratio = np.abs(needed) / qd_max # fracción de límite consumida
    bottleneck = np.max(ratio)
    return 1.0 / bottleneck if bottleneck > 1e-12 else np.inf


# Trayectoria de muestra: y=0, z=0.70, avance en x de 0.35 a 0.55
p0 = np.array([0.35, 0.0, 0.70])
p1 = np.array([0.55, 0.0, 0.70])
L = np.linalg.norm(p1 - p0)
d_hat = (p1 - p0) / L
N = 50
puntos = np.linspace(p0, p1, N)

v_max_path = []
sigma_min_path = []

q_actual = np.array([0.0, -0.3, 0.5])  # semilla inicial

for p_obj in puntos:
    sol = robot.ikine_LM(
        sm.SE3(*p_obj), q0=q_actual, mask=[1, 1, 1, 0, 0, 0], joint_limits=True
    )
    if not sol.success:
        v_max_path.append(np.nan)
        sigma_min_path.append(np.nan)
        continue
    q_actual = np.asarray(sol.q)
    Jpos = np.asarray(robot.jacob0(q_actual)[:3, :3])
    svs = np.linalg.svd(Jpos, compute_uv=False)
    sigma_min_path.append(float(svs[-1]))
    v_max_path.append(v_max_en_configuracion(robot, q_actual, d_hat, QD_MAX))

v_max_path = np.array(v_max_path)
sigma_min_path = np.array(sigma_min_path)
s_vals = np.linspace(0, 1, N)

v_sostenible = np.nanmin(v_max_path)
T_opt = L / v_sostenible if v_sostenible > 0 else np.inf

print(f"Longitud del segmento: L = {L:.3f} m")
print(f"Dirección: d̂ = {d_hat}")
print(f"v* (velocidad sostenible) = {v_sostenible:.4f} m/s")
print(f"T* (tiempo mínimo)        = {T_opt:.4f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# — Panel izquierdo: v_max a lo largo del segmento
ax = axes[0]
ax.plot(s_vals, v_max_path, color="tab:blue", linewidth=2, label="$v_{max}(s)$")
ax.axhline(
    v_sostenible,
    color="tab:red",
    linestyle="--",
    linewidth=1.5,
    label=f"$v^*$ = {v_sostenible:.3f} m/s (cuello de botella)",
)
ax.set_xlabel("$s$ (posición normalizada a lo largo del segmento)")
ax.set_ylabel("$v_{max}$ [m/s]")
ax.set_title("Velocidad cartesiana máxima factible a lo largo del segmento")
ax.legend()
ax.grid(True, alpha=0.3)

# — Panel derecho: sigma_min a lo largo del segmento
ax = axes[1]
ax.plot(
    s_vals, sigma_min_path, color="tab:green", linewidth=2
)
ax.set_xlabel("$s$ (posición normalizada)")
ax.set_ylabel("$\\sigma_{min}(J_{pos})$ [m/rad]")
ax.set_title("Valor singular mínimo del Jacobiano posicional")
ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Trayectoria muestra: $y=0$, $z=0.70$ m  |  $L={L:.2f}$ m, $\\hat{{d}}=\\hat{{x}}$",
    fontsize=11,
)
fig.tight_layout()
plt.show()

In [ ]:
# Verificación de la cota teórica:
#   v* <= sigma_min_min * min(qd_max)
# (válida para límites isótropos; aquí los límites NO son isótropos, es una cota)
sigma_min_min = float(np.nanmin(sigma_min_path))
qd_min_max = float(np.min(QD_MAX))
cota_teorica = sigma_min_min * qd_min_max

print(f"sigma_min mínimo en la trayectoria : {sigma_min_min:.6f} m/rad")
print(f"qd_max mínimo (eje 1 ó 2)          : {np.rad2deg(qd_min_max):.1f} °/s  ({qd_min_max:.4f} rad/s)")
print(f"Cota teórica v* <= sigma*qd_min    : {cota_teorica:.4f} m/s")
print(f"v* numérico real                   : {v_sostenible:.4f} m/s")
print(f"¿Cota satisfecha? {v_sostenible <= cota_teorica + 1e-9}")